<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/OCR-YOLO/blob/main/roboflow%2BOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.3/260.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 140.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92


In [3]:


from roboflow import Roboflow
rf = Roboflow(api_key="czJ7BJ4MVFLxiWc69wtu")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(8)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-8 in yolov8:: 100%|██████████| 112/112 [00:00<00:00, 6644.91it/s]


In [4]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.6 MB/s eta 0:00:00


In [5]:
from ultralytics import YOLO
import shutil,os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [11]:
model=YOLO("yolov8n.pt")

results=model.train(data=dataset.location + "/data.yaml",
                    epochs=5,imgsz=640,batch=16,
                    workers=2,device=0)

os.makedirs("saved_models",exist_ok=True)
shutil.copy("runs/detect/train/weights/best.pt","saved_models/license_plate_best.pt")
shutil.copy("runs/detect/train/weights/last.pt","saved_models/license_plate_last.pt")
print("weights saved in saved_models/")

Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/License-Plate-Recognition-8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [8]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 31.3 MB/s eta 0:00:00


In [9]:
import cv2
import numpy as np
import easyocr
import re
from collections import defaultdict, deque

In [ ]:
model=YOLO("license_plate_best.pt")
reader=easyocr.Reader(['en'],gpu=True)

plate_pattern=re.compile(r"[A-Z]{2}[0-9]{2}[A-Z]{3}$")

In [ ]:
def correct_plate_format(ocr_text):
  mapping_num_to_alpha={"0":"O","1":"I","5":"S","8":"B"}
  mapping_alpha_to_num={"O":"0","I":"1","Z":"2","S":"5","B":"8"}

  ocr_text=ocr_text.upper().replace(" ","")
  if len(ocr_text)!=7:
    return ""

  corrected=[]
  for i,ch in enumerate(ocr_text):
    if i<2 or i>=4:
      if ch.isdigit() and ch in mapping_num_to_alpha:
        corrected.append(mapping_num_to_alpha[ch])
      elif ch.alpha():
        corrected.append(ch)
      else:
        return ""

    else:
      if ch.isalpha() and ch in mapping_alpha_to_num:
        corrected.append(mapping_alpha_to_num[ch])
      elif ch.isdigit():
        corrected.append(ch)
      else:
        return ""

  return "".join(corrected)

In [ ]:
def recognize(plate):
  if plate.size==0:
    return ""

  gray=cv2.cvtColor(plate,cv2.COLOR_BGR@GRAY)
  _,thresh=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
  plate_resized=cv2.resize(thresh, None, fx=2,fy=2,interpolation=cv2.INTER_CUBIC)

  try:
    ocr_result=reader.readtext(plate_resized,detail=0,allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

    if len(ocr_result)>0:
      candidate=correct_plate_format(ocr_result[0])
      if candidate and plate_pattern.match(candidate):
        return candidate

  except:
    pass


  return ""


In [ ]:
plate_history=defaultdict(lambda:deque(maxlen=10))
plate_final={}

def get_box_id(x1,y1,x2,y2):
  return f"{int(x1/10)}_{int(y1/10)}_{int(x2/10)}_{int(y2/10)}"

def stable_plate(box_id,new text):
  if new_text:
    plate_history[box_id].append(new_text)

    most_common=max(set(plate_history[box_id]),key=plate_history[box_id].count)
    plate_final[box_id]=most_common

    return plate_final.get(box_id,"")

In [ ]:
input_video="vechile_video.mp4"
output_video="output_with_licensev3.mp4"

cap=cv2.VideoCapture(input_video)
fourcc=cv2.VideoWriter_fourcc(*'mp4v')
out=cv2.VideoWriter(output_video,fourcc,cap.get(cv2.CAP_PROP_FPS),
                    (int(cap.get(3)),int(cap.get(4))))

conf_thresh=0.3

In [ ]:
while cap.isOpened():
  ret,frame=cap.read()
  if not ret:
    break

  results=model(frame,verbose=False)

  for r in results:
    boxes=r.boxes
    for box in boxes:
      conf=float(box.conf.cpu().numpy()[0])
      if conf<CONF_THRESH:
        continue

        x1,y1,x2,y2=map(int,box.xyxy.cpu().numpy()[0])
        plate=frame[y1:y2,x1:x2]

        text=recongnize(plate)
        box_id=get_box_id(x1,y1,x2,y2)
        stable_text=stable_plate()
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),3)

        if plate.size>0:
          overlay_h,overlay_w=150,400
          plate_resized=cv2.resize(plate,(overlay_w, overlay_h))

          oy1=max(0,y1-overlay_h-40)
          ox1=x1
          oy2,ox2=oy1+overlay_h,ox1+overlay_w

          if oy2<=frame.shape[0] and ox2<=frame.shape[1]:
            frame[oy1:oy2,ox1:ox2]=plate_resized

            if stable_text:
              cv2.putText(frame, stable_text, (ox1,oy1-20),
                          cv2.FONT_HERSHEY_SIMPLEX,2,(0,0,0),6)
              cv2.putText(frame, stable_text, (ox1,oy1-20),
                          cv2.FONT_HERSHEY_SIMPLEX,2,(255,255,255),3)

  out.write(frame)
  cv2.imshow("Annotated Video",frame)
  if cv2.waitKey(1) & 0xFF == ord('q'):
   break

In [ ]:
cap.release()
out.release()
cv2.destroyAllWindows()

print("video saved", output_video)